# 检查 `invoice_panel.dta` 企业数为什么从 3376 变成 2108

这个 notebook 只做诊断，不会覆盖任何正式数据文件。

目标：逐步复现 `01_clean.ipynb` 中从原始 CSV 到 `dv` / `invoice_panel` 的关键步骤，并和当前磁盘上的 `invoice_panel.dta` 对比。

重点检查：
1. 原始样本企业数是不是 3410；
2. `firm_buy` / `firm_sell` 清洗后分别剩多少企业；
3. 只保留合法 9 位产品码后剩多少企业；
4. 定义外包产品，即同一企业同一产品既买又卖后，剩多少企业；
5. 构造出来的临时 `dv_check` 和当前磁盘 `invoice_panel.dta` 是否一致。


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 220)

BASE = Path(r'G:\Kuangyu_Temp\Outsource\productivity')
ROOT = BASE.parent

print('BASE =', BASE)
print('ROOT =', ROOT)
print('invoice_panel exists:', (BASE / 'invoice_panel.dta').exists())

## 1. 先看当前磁盘上的 `invoice_panel.dta`

这一步确认当前 `02_price_reg.do` 实际读到的文件规模。

In [ ]:
invoice_path = BASE / 'invoice_panel.dta'
disk_inv = pd.read_stata(invoice_path, convert_categoricals=False)

print('current invoice_panel.dta rows:', f'{len(disk_inv):,}')
print('current invoice_panel.dta unique firms:', f"{disk_inv['firm_id'].nunique():,}")
print('current invoice_panel.dta unique products:', f"{disk_inv['product_id'].nunique():,}")
print('current invoice_panel.dta unique cities:', f"{disk_inv['city'].nunique():,}")
display(disk_inv.head())

## 2. 读入原始 CSV，检查原始企业数

这里对应 `01_clean.ipynb` 的 Step 1。

In [ ]:
NA = ['', 'NULL', '(Null)', 'null', 'None', 'nan', 'NaN', '#N/A']

firm_buy  = pd.read_csv(BASE / 'firm_buy.csv',  dtype=str, na_values=NA, encoding='utf-8-sig')
firm_sell = pd.read_csv(BASE / 'firm_sell.csv', dtype=str, na_values=NA, encoding='utf-8-sig')
firm_city = pd.read_csv(BASE / 'firm_city.csv', dtype=str, na_values=NA, encoding='utf-8-sig')

for df in [firm_buy, firm_sell, firm_city]:
    df.columns = df.columns.str.strip()

raw_buy_firms = firm_buy['购方企业ID'].astype('string').str.strip().nunique()
raw_sell_firms = firm_sell['销方企业ID'].astype('string').str.strip().nunique()
raw_city_firms = firm_city['企业ID'].astype('string').str.strip().nunique()

print('firm_buy rows:', f'{len(firm_buy):,}', 'unique buyer firms:', f'{raw_buy_firms:,}')
print('firm_sell rows:', f'{len(firm_sell):,}', 'unique seller firms:', f'{raw_sell_firms:,}')
print('firm_city rows:', f'{len(firm_city):,}', 'unique firms:', f'{raw_city_firms:,}')

buy_set = set(firm_buy['购方企业ID'].astype('string').str.strip().dropna())
sell_set = set(firm_sell['销方企业ID'].astype('string').str.strip().dropna())
city_set = set(firm_city['企业ID'].astype('string').str.strip().dropna())

print('buy ∩ sell firms:', f'{len(buy_set & sell_set):,}')
print('city firms not in buy:', f'{len(city_set - buy_set):,}')
print('city firms not in sell:', f'{len(city_set - sell_set):,}')

## 3. 合并地区，统一列名

这里检查地区匹配是否导致企业减少。注意 merge 是 left merge，理论上不会减少行数。

In [ ]:
firm_city_map = firm_city.rename(columns={'企业ID': 'firm_id', '地区': 'city'})[['firm_id', 'city']].copy()
firm_city_map['firm_id'] = firm_city_map['firm_id'].astype('string').str.strip()
firm_city_map['city'] = firm_city_map['city'].astype('string').str.strip()

fb = firm_buy.rename(columns={'购方企业ID': 'firm_id', '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'})[['firm_id', 'product_code', 'value', 'qty']].copy()
fb['firm_id'] = fb['firm_id'].astype('string').str.strip()
fb = fb.merge(firm_city_map, on='firm_id', how='left', validate='m:1')
fb = fb[['firm_id', 'city', 'product_code', 'value', 'qty']].copy()

fs = firm_sell.rename(columns={'销方企业ID': 'firm_id', '项目代码': 'product_code', '金额合计': 'value', '数量合计': 'qty'})[['firm_id', 'product_code', 'value', 'qty']].copy()
fs['firm_id'] = fs['firm_id'].astype('string').str.strip()
fs = fs.merge(firm_city_map, on='firm_id', how='left', validate='m:1')
fs = fs[['firm_id', 'city', 'product_code', 'value', 'qty']].copy()

print('fb after city merge rows:', f'{len(fb):,}', 'firms:', f"{fb['firm_id'].nunique():,}", 'missing city rows:', f"{fb['city'].isna().sum():,}")
print('fs after city merge rows:', f'{len(fs):,}', 'firms:', f"{fs['firm_id'].nunique():,}", 'missing city rows:', f"{fs['city'].isna().sum():,}")

## 4. 清洗金额、数量、项目代码，生成 9 位 `product_id`

这里对应 `01_clean.ipynb` Step 3。会删除金额/数量缺失、非纯数字项目代码、金额或数量非正的行。

In [ ]:
stage_rows = []

stage_rows.append({'stage': 'raw firm_city', 'rows': len(firm_city), 'firms': firm_city_map['firm_id'].nunique(), 'products': np.nan})
stage_rows.append({'stage': 'fb after city merge', 'rows': len(fb), 'firms': fb['firm_id'].nunique(), 'products': fb['product_code'].nunique()})
stage_rows.append({'stage': 'fs after city merge', 'rows': len(fs), 'firms': fs['firm_id'].nunique(), 'products': fs['product_code'].nunique()})

fb0 = fb.copy()
fs0 = fs.copy()

fb['value'] = pd.to_numeric(fb['value'], errors='coerce')
fb['qty'] = pd.to_numeric(fb['qty'], errors='coerce')
fb = fb.dropna(subset=['product_code', 'value', 'qty']).copy()
fb = fb[fb['product_code'].str.fullmatch(r'\d+', na=False)].copy()
fb['product_id'] = fb['product_code'].str.ljust(19, '0').str[:9]
fb = fb[(fb['value'] > 0) & (fb['qty'] > 0)].copy()

fs['value'] = pd.to_numeric(fs['value'], errors='coerce')
fs['qty'] = pd.to_numeric(fs['qty'], errors='coerce')
fs = fs.dropna(subset=['product_code', 'value', 'qty']).copy()
fs = fs[fs['product_code'].str.fullmatch(r'\d+', na=False)].copy()
fs['product_id'] = fs['product_code'].str.ljust(19, '0').str[:9]
fs = fs[(fs['value'] > 0) & (fs['qty'] > 0)].copy()

stage_rows.append({'stage': 'fb after numeric/code/positive', 'rows': len(fb), 'firms': fb['firm_id'].nunique(), 'products': fb['product_id'].nunique()})
stage_rows.append({'stage': 'fs after numeric/code/positive', 'rows': len(fs), 'firms': fs['firm_id'].nunique(), 'products': fs['product_id'].nunique()})

display(pd.DataFrame(stage_rows))

## 5. 只保留合法 9 位产品码 `bianma`

这里对应 `01_clean.ipynb` Step 4。

In [ ]:
bianma_path_1 = ROOT / 'bianma.dta'
bianma_path_2 = BASE / 'bianma.dta'
bianma_path = bianma_path_1 if bianma_path_1.exists() else bianma_path_2
print('using bianma:', bianma_path)

bianma = pd.read_stata(bianma_path, convert_categoricals=False)
valid = set(bianma['product_id'].astype(str).str.strip())
print('valid 9-digit products:', f'{len(valid):,}')

fb_before_bianma = fb.copy()
fs_before_bianma = fs.copy()

fb = fb[fb['product_id'].isin(valid)].copy()
fs = fs[fs['product_id'].isin(valid)].copy()

stage_rows.append({'stage': 'fb after bianma', 'rows': len(fb), 'firms': fb['firm_id'].nunique(), 'products': fb['product_id'].nunique()})
stage_rows.append({'stage': 'fs after bianma', 'rows': len(fs), 'firms': fs['firm_id'].nunique(), 'products': fs['product_id'].nunique()})

display(pd.DataFrame(stage_rows))

fb_firms_before = set(fb_before_bianma['firm_id'].dropna())
fb_firms_after = set(fb['firm_id'].dropna())
fs_firms_before = set(fs_before_bianma['firm_id'].dropna())
fs_firms_after = set(fs['firm_id'].dropna())
print('fb firms lost by bianma:', f'{len(fb_firms_before - fb_firms_after):,}')
print('fs firms lost by bianma:', f'{len(fs_firms_before - fs_firms_after):,}')

## 6. 识别外包产品对：同一企业同一产品既买又卖

这是最可能让企业数下降的核心步骤。因为只有出现在 `fb_agg` 和 `fs_agg` 交集中的 firm-product 才会进入外包样本。

In [ ]:
fb_agg = fb.groupby(['firm_id', 'product_id'], as_index=False).agg(buy_value=('value', 'sum'))
fs_agg = fs.groupby(['firm_id', 'product_id'], as_index=False).agg(sell_value=('value', 'sum'))

both = fb_agg.merge(fs_agg, on=['firm_id', 'product_id'], how='inner')
both['outsourcing_value'] = both[['buy_value', 'sell_value']].min(axis=1)
outsourcing_pairs = both[both['outsourcing_value'] > 0][['firm_id', 'product_id']].copy()

stage_rows.append({'stage': 'fb_agg firm-product', 'rows': len(fb_agg), 'firms': fb_agg['firm_id'].nunique(), 'products': fb_agg['product_id'].nunique()})
stage_rows.append({'stage': 'fs_agg firm-product', 'rows': len(fs_agg), 'firms': fs_agg['firm_id'].nunique(), 'products': fs_agg['product_id'].nunique()})
stage_rows.append({'stage': 'both buy and sell same product', 'rows': len(both), 'firms': both['firm_id'].nunique(), 'products': both['product_id'].nunique()})
stage_rows.append({'stage': 'outsourcing_pairs', 'rows': len(outsourcing_pairs), 'firms': outsourcing_pairs['firm_id'].nunique(), 'products': outsourcing_pairs['product_id'].nunique()})

display(pd.DataFrame(stage_rows))

fb_firms = set(fb['firm_id'].dropna())
out_firms = set(outsourcing_pairs['firm_id'].dropna())
print('fb firms after bianma:', f'{len(fb_firms):,}')
print('firms with at least one outsourced product pair:', f'{len(out_firms):,}')
print('fb firms lost because no same product appears on sell side:', f'{len(fb_firms - out_firms):,}')

## 7. 过滤采购明细到外包产品，并构造 `dv_check`

这一步复现 `01_clean.ipynb` Step 4c 和 Step 5，但不保存文件。

In [ ]:
fb_before_out_filter = fb.copy()
fb_out = fb.merge(outsourcing_pairs, on=['firm_id', 'product_id'], how='inner')

print('fb before outsourcing filter rows:', f'{len(fb_before_out_filter):,}', 'firms:', f"{fb_before_out_filter['firm_id'].nunique():,}")
print('fb after outsourcing filter rows:', f'{len(fb_out):,}', 'firms:', f"{fb_out['firm_id'].nunique():,}", 'products:', f"{fb_out['product_id'].nunique():,}")

fb_out['year'] = 2017
dv_check = fb_out.groupby(['firm_id', 'product_id', 'city', 'year'], as_index=False).agg(
    value=('value', 'sum'),
    qty=('qty', 'sum'),
    n_rows=('value', 'size')
)
dv_check = dv_check[(dv_check['value'] > 0) & (dv_check['qty'] > 0)].copy()
dv_check['p_buy'] = dv_check['value'] / dv_check['qty']
dv_check['ln_p_buy'] = np.log(dv_check['p_buy'])
dv_check['ln_p_net'] = dv_check['ln_p_buy']
dv_check['p_net'] = dv_check['p_buy']
dv_check['ln_qty'] = np.log(dv_check['qty'])

print('dv_check rows:', f'{len(dv_check):,}')
print('dv_check unique firms:', f"{dv_check['firm_id'].nunique():,}")
print('dv_check unique products:', f"{dv_check['product_id'].nunique():,}")
print('dv_check unique cities:', f"{dv_check['city'].nunique():,}")
display(dv_check[['p_buy', 'qty', 'value']].describe(percentiles=[.01, .05, .5, .95, .99]).round(2))

## 8. 对比重新计算的 `dv_check` 和当前磁盘 `invoice_panel.dta`

如果两者不一致，说明当前磁盘 `invoice_panel.dta` 不是由当前这套 Step 1–5 逻辑直接保存出来的，或者 notebook 中间有其他未展示/未按顺序运行的过滤。

In [ ]:
key_cols = ['firm_id', 'product_id', 'city', 'year']

disk_keys = disk_inv[key_cols].copy()
check_keys = dv_check[key_cols].copy()

for col in ['firm_id', 'product_id', 'city']:
    disk_keys[col] = disk_keys[col].astype('string').str.strip()
    check_keys[col] = check_keys[col].astype('string').str.strip()

disk_keys['year'] = pd.to_numeric(disk_keys['year'], errors='coerce').astype('Int64')
check_keys['year'] = pd.to_numeric(check_keys['year'], errors='coerce').astype('Int64')

cmp = check_keys.merge(disk_keys.drop_duplicates(), on=key_cols, how='outer', indicator=True)
print(cmp['_merge'].value_counts())

print('dv_check rows:', f'{len(dv_check):,}', 'firms:', f"{dv_check['firm_id'].nunique():,}")
print('disk invoice rows:', f'{len(disk_inv):,}', 'firms:', f"{disk_inv['firm_id'].nunique():,}")

left_only_firms = cmp.loc[cmp['_merge'] == 'left_only', 'firm_id'].nunique()
right_only_firms = cmp.loc[cmp['_merge'] == 'right_only', 'firm_id'].nunique()
print('firms in dv_check but not disk invoice:', f'{left_only_firms:,}')
print('firms in disk invoice but not dv_check:', f'{right_only_firms:,}')

display(cmp['_merge'].value_counts(normalize=True).rename('share').to_frame())

## 9. 检查被当前 `invoice_panel.dta` 排除的企业有什么特征

如果 `dv_check` 大于磁盘文件，这里列出在 `dv_check` 中存在、但磁盘 `invoice_panel.dta` 中不存在的企业数量和产品数量分布。

In [ ]:
disk_firms = set(disk_inv['firm_id'].astype('string').str.strip().dropna())
check_firms = set(dv_check['firm_id'].astype('string').str.strip().dropna())

missing_from_disk = sorted(check_firms - disk_firms)
extra_in_disk = sorted(disk_firms - check_firms)

print('firms in dv_check:', f'{len(check_firms):,}')
print('firms in disk invoice:', f'{len(disk_firms):,}')
print('firms in dv_check but missing from disk invoice:', f'{len(missing_from_disk):,}')
print('firms in disk invoice but not in dv_check:', f'{len(extra_in_disk):,}')

tmp_missing = dv_check[dv_check['firm_id'].astype('string').str.strip().isin(missing_from_disk)].copy()
if len(tmp_missing) > 0:
    missing_summary = tmp_missing.groupby('firm_id').agg(
        rows=('product_id', 'size'),
        products=('product_id', 'nunique'),
        cities=('city', 'nunique'),
        value=('value', 'sum'),
        qty=('qty', 'sum')
    ).sort_values('rows', ascending=False)
    display(missing_summary.describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]).round(2))
    display(missing_summary.head(20))
else:
    print('No missing firms from disk invoice.')

## 10. 结论提示

看上面几块输出即可定位问题：

- 如果 `dv_check` 是 403,400 行、3,376 家，但磁盘 `invoice_panel.dta` 是 59,445 行、2,108 家：说明当前磁盘文件和当前 `01_clean.ipynb` Step 1–5 逻辑不一致，需要检查最后保存 cell 是否真的运行，或者是否有其他脚本/旧版本覆盖了 `invoice_panel.dta`。
- 如果 `dv_check` 本身就是 59,445 行、2,108 家：说明企业数下降发生在 Step 1–5 内部，上面的 stage table 会显示具体是哪一步。
- 如果主要下降发生在 `outsourcing_pairs`：说明 3,410 家里面很多企业没有“同一企业同一产品既买又卖”的外包产品，因此不会进入外包采购价格面板。
